# circuit_lm — Train + Evaluate + Trace

Full pipeline: train a PDA circuit + neural corrector hybrid on your data,
then trace through predictions to see *why* the model made each choice.

**Runtime**: GPU recommended for corrector training (Colab T4/V100). Circuit training is CPU-fast.

**Estimated time**: ~5-10 min for full pipeline on 1MB data.

## 1. Setup — clone repo + install deps

In [ ]:
!git clone https://github.com/toxzak-svg/circuit_lm.git 2>&1 | tail -3
%cd circuit_lm
!pip install -q ortools pandas 2>&1 | tail -3

## 2. Upload your training data

Upload a `.txt` file of plain text. Format: raw text with paragraphs separated by blank lines.
For chat data use `User:` / `Assistant:` prefix format.

In [ ]:
from google.colab import files
uploaded = files.upload()
data_path = list(uploaded.keys())[0]
print(f"Uploaded: {data_path}")

## 3. Train Circuit (FSM or PDA)

In [ ]:
import subprocess, json, time

# Config — edit these
AUTOMATON = "pda"          # "fsm", "pda", or "ppm"
VOCAB_SIZE = 1024
STATE_BITS = 5             # 2^5 = 32 states
STACK_DEPTH = 6            # PDA stack depth
BPE_MERGES = 512           # BPE vocabulary size
CIRCUIT_OUT = "circuit.json"

cmd = [
    "python", "-m", "circuit_lm.cli", "train",
    "--data", data_path,
    "--out", CIRCUIT_OUT,
    "--automaton", AUTOMATON,
    "--vocab_size", str(VOCAB_SIZE),
    "--state_bits", str(STATE_BITS),
    "--stack_depth", str(STACK_DEPTH),
    "--tokenizer", "bpe",
    "--bpe_merges", str(BPE_MERGES),
    "--steps", "60",
]
print(f"Running: {' '.join(cmd)}")
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Training failed")
print(f"Circuit trained in {time.time()-t0:.1f}s")

## 4. Train Hybrid Corrector (Neural Net)

In [ ]:
# Copy training data into circuit_lm dir so hybrid train can find it
import shutil
shutil.copy(data_path, data_path)  # already in cwd

CORRECTOR_OUT = "corrector.pt"

cmd = [
    "python", "scripts/train_bpe_hybrid.py",
    "--data", data_path,
    "--circuit-out", CIRCUIT_OUT,
    "--corrector-out", CORRECTOR_OUT,
    "--automaton", AUTOMATON,
    "--vocab-size", str(VOCAB_SIZE),
    "--bpe-merges", str(BPE_MERGES),
    "--state-bits", str(STATE_BITS),
    "--stack-depth", str(STACK_DEPTH),
    "--epochs", "5",
    "--batch-size", "128",
    "--max-examples", "100000",
    "--embed-dim", "256",
    "--hidden-dim", "512",
    "--num-layers", "3",
]
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Corrector training failed")
print(f"\nCorrector trained in {time.time()-t0:.1f}s")

## 5. Trace — See Why Each Token Was Predicted

In [ ]:
# Trace through a prompt and see state + stack + top-k predictions at each step
PROMPT = "User: hello\nAssistant:"  # try your own prompt here
TOP_K = 5

cmd = [
    "python", "-m", "circuit_lm.cli", "trace",
    "--model", CIRCUIT_OUT,
    "--prompt", PROMPT,
    "--top_k", str(TOP_K),
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

### What you're seeing:

- **step**: position in the token sequence
- **token**: the actual next token (ground truth)
- **state**: the FSM/PDA state the model is in before seeing this token
- **stack_top**: PDA stack top (-1 = empty, shown for PDA only)
- **top_k**: model's ranked predictions for what comes next

If the gold token is NOT in top_k, the model made an error at that step.

## 6. Evaluate Accuracy

In [ ]:
# Split off some data for eval
import random, pathlib

all_text = pathlib.Path(data_path).read_text()
lines = [l for l in all_text.split('\n') if l.strip()]
random.seed(42)
random.shuffle(lines)
split = int(len(lines) * 0.9)
train_text = '\n'.join(lines[:split])
eval_text = '\n'.join(lines[split:])

eval_path = "eval_data.txt"
pathlib.Path(eval_path).write_text(eval_text)

cmd = [
    "python", "-m", "circuit_lm.cli", "eval",
    "--model", CIRCUIT_OUT,
    "--data", eval_path,
    "--per_token",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

## 7. Download Trained Models

In [ ]:
files.download(CIRCUIT_OUT)
files.download(CORRECTOR_OUT)
print("Download circuit.json and corrector.pt to use with the CLI:")
print(f"  py -3.12 -m circuit_lm.cli chat --model {CIRCUIT_OUT} --corrector {CORRECTOR_OUT}")